# Punctuality and cancellation status

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

In [2]:
data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_operators.parquet"

try: 
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)

data_chuuchuu.shape

(7035704, 36)

In [3]:
data_chuuchuu["normalized_operator"].value_counts(dropna=False)

normalized_operator
SNCF                                       6420840
SNCF VOYAGEURS                              337966
Eurostar                                    153926
Deutsche Bahn                                62093
Trenitalia                                   24845
SNCF Voyageurs LO                            21041
OCEdefault                                   12469
Conseil Régional Auvergne - Rhône-Alpes       1727
SNCF Voyageurs EA                              718
SNCF Voyageurs SA                               79
Name: count, dtype: int64

In [4]:
data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="t"]["operator"].unique()

<ArrowStringArray>
[                                   'SNCF',
                                'Eurostar',
                           'Deutsche Bahn',
                          'SNCF VOYAGEURS',
 'Conseil Régional Auvergne - Rhône-Alpes',
                       'SNCF Voyageurs LO',
                       'SNCF Voyageurs SA']
Length: 7, dtype: str

Checking which rows has / has no cancellation data

In [5]:
arr_cancelled_t = len(data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="t"])/len(data_chuuchuu["arrivalCancelled"])*100
arr_cancelled_f = len(data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="f"])/len(data_chuuchuu["arrivalCancelled"])*100

dep_cancelled_t = len(data_chuuchuu[data_chuuchuu["departureCancelled"]=="t"])/len(data_chuuchuu["departureCancelled"])*100
dep_cancelled_f = len(data_chuuchuu[data_chuuchuu["departureCancelled"]=="f"])/len(data_chuuchuu["departureCancelled"])*100

print("Share of row arrivalCancelled = t")
print(round(arr_cancelled_t,2),"%")
print("Share of row arrivalCancelled = f")
print(round(arr_cancelled_f,2),"%")
print("Share of row without data cancelation")
print(round(100-(arr_cancelled_f+arr_cancelled_t),2),"%")
print("#######")

print("Share of row departureCancelled = t")
print(round(dep_cancelled_t,2),"%")
print("Share of row departureCancelled = f")
print(round(dep_cancelled_f,2),"%")
print("Share of row without data cancelation")
print(round(100-(dep_cancelled_f+dep_cancelled_t),2),"%")

Share of row arrivalCancelled = t
1.67 %
Share of row arrivalCancelled = f
95.64 %
Share of row without data cancelation
2.69 %
#######
Share of row departureCancelled = t
1.67 %
Share of row departureCancelled = f
95.64 %
Share of row without data cancelation
2.69 %


In [6]:
data_chuuchuu[data_chuuchuu["depart_terminus"]=="depart"]["departureCancelled"].value_counts(dropna=False)

departureCancelled
f      842909
t       14871
NaN        10
Name: count, dtype: int64

### Recovering a cancellation status for rows where `arrivalCancelled` is null

Per the terminology doc: *"If null and a delay was recorded, you can assume the arrival was not cancelled."*

But actually it seems that most arrivalCancelled rows with a null value are departing stations. To be sure that the train is not cancelled, we need to check if `departure` is null or not



In [7]:
arr_null_cancelled = data_chuuchuu["arrivalCancelled"].isna()

# among the rows with no explicit flag, an effective departure timestamp means it wasn't cancelled
arr_inferred_not_cancelled = arr_null_cancelled & data_chuuchuu["departure"].notna()

# still no way to know -- neither an explicit flag nor an effective departure to infer from
arr_unreliable = arr_null_cancelled & data_chuuchuu["departure"].isna()

n = len(data_chuuchuu)
arr_share_explicit = (~arr_null_cancelled).sum() / n * 100
arr_share_inferred = arr_inferred_not_cancelled.sum() / n * 100
arr_share_unreliable = arr_unreliable.sum() / n * 100

print(f"Share of rows with an explicit arrivalCancelled value (t/f): {arr_share_explicit:.2f}%")
print(f"Share of rows with arrivalCancelled null but inferred not cancelled (dparture is not null): {arr_share_inferred:.2f}%")
print(f"Share of rows with no reliable cancellation data at all: {arr_share_unreliable:.2f}%")
print()
print(f"Total reliable cancellation data: {arr_share_explicit + arr_share_inferred:.2f}%")

Share of rows with an explicit arrivalCancelled value (t/f): 97.31%
Share of rows with arrivalCancelled null but inferred not cancelled (dparture is not null): 2.69%
Share of rows with no reliable cancellation data at all: 0.00%

Total reliable cancellation data: 100.00%


In [8]:
data_chuuchuu["arrivalCancelled_resolved"] = data_chuuchuu["arrivalCancelled"]
data_chuuchuu.loc[arr_inferred_not_cancelled, "arrivalCancelled_resolved"] = "f"

data_chuuchuu["arrivalCancelled_resolved"].value_counts(dropna=False)

arrivalCancelled_resolved
f    6918153
t     117551
Name: count, dtype: int64

In [9]:
arr_unreliable_rows = data_chuuchuu[arr_unreliable]
arr_unreliable_rows

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,plannedArrival,arrivalDelay,arrivalPlatform,plannedArrivalPlatform,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,operator,uicCodeStop,country,stopName_slug,sort_time,sort_time_source,journey_id,is_ambiguous_trip,journey_verificator,is_cross_agency_duplicate,cross_agency_duplicate_confidence,depart_terminus,journey_type,normalized_operator,arrivalCancelled_resolved


# Now same departure Cancelled

Step 1: if departutreCancelled = Na, but stop = Terminus: ok

In [10]:
dep_null_cancelled = data_chuuchuu["departureCancelled"].isna()

# among the rows with no explicit flag, if we are at the terminus it mean there is no issue
dep_inferred_not_cancelled = dep_null_cancelled & (data_chuuchuu["depart_terminus"]=="terminus")

# still no way to know -- neither an explicit flag and we are not at terminus
dep_unreliable = dep_null_cancelled & (data_chuuchuu["depart_terminus"] != "terminus")

n = len(data_chuuchuu)
dep_share_explicit = (~dep_null_cancelled).sum() / n * 100
dep_share_inferred = dep_inferred_not_cancelled.sum() / n * 100
share_dep_unreliable = dep_unreliable.sum() / n * 100

print(f"Share of rows with an explicit departureCancelled value (t/f): {dep_share_explicit:.2f}%")
print(f"Share of rows with departureCancelled null but inferred not cancelled (dparture is not null): {dep_share_inferred:.2f}%")
print(f"Share of rows with no reliable cancellation data at all: {share_dep_unreliable:.2f}%")
print()
print(f"Total reliable cancellation data: {dep_share_explicit + dep_share_inferred:.2f}%")

Share of rows with an explicit departureCancelled value (t/f): 97.31%
Share of rows with departureCancelled null but inferred not cancelled (dparture is not null): 2.65%
Share of rows with no reliable cancellation data at all: 0.03%

Total reliable cancellation data: 99.97%


In [11]:
data_chuuchuu["departureCancelled_resolved"] = data_chuuchuu["departureCancelled"]
data_chuuchuu.loc[dep_inferred_not_cancelled, "departureCancelled_resolved"] = "f"

data_chuuchuu["departureCancelled_resolved"].value_counts(dropna=False)

departureCancelled_resolved
f      6915810
t       117551
NaN       2343
Name: count, dtype: int64

In [12]:
dep_unreliable_rows = data_chuuchuu[dep_unreliable]
dep_unreliable_rows.head(5)

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,plannedArrival,arrivalDelay,arrivalPlatform,plannedArrivalPlatform,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,operator,uicCodeStop,country,stopName_slug,sort_time,sort_time_source,journey_id,is_ambiguous_trip,journey_verificator,is_cross_agency_duplicate,cross_agency_duplicate_confidence,depart_terminus,journey_type,normalized_operator,arrivalCancelled_resolved,departureCancelled_resolved
1303,FR,TGV INOUI,6685,2025-01-01,8772600,2025-01-01 14:44:58.571+00,TGV INOUI 6685,StopPoint:OCETGV INOUI-87726000,Saint-Étienne Châteaucreux,2025-01-01 14:48:00+00:00,2025-01-01 14:48:00+00:00,0.0,NaN,NaN,f,NaT,NaT,NaN,NaN,NaN,NaN,{},SNCF,87,France,saint-etienne-chateaucreux,2025-01-01 14:48:00+00:00,arrival,FR_TGV INOUI_6685_2025-01-01,False,TGV INOUI_6685_2025-01-01_8772600,False,not_flagged,intermediate,domestic,SNCF,f,NaN
1533,FR,ICE,9553,2025-01-01,8011068,2025-01-01 15:54:47.207+00,ICE 9553,StopPoint:OCEICE-80110684,Francfort sur le Main,2025-01-01 15:59:00+00:00,2025-01-01 15:59:00+00:00,0.0,NaN,NaN,f,NaT,NaT,NaN,NaN,NaN,NaN,{},Deutsche Bahn,80,Germany,francfort-sur-le-main,2025-01-01 15:59:00+00:00,arrival,FR_ICE_9553_2025-01-01,False,ICE_9553_2025-01-01_8011068,False,not_flagged,intermediate,international,Deutsche Bahn,f,NaN
2070,FR,ICE,9555,2025-01-01,8011068,2025-01-01 18:05:37.035+00,ICE 9555,StopPoint:OCEICE-80110684,Francfort sur le Main,2025-01-01 19:59:00+00:00,2025-01-01 19:59:00+00:00,0.0,NaN,NaN,f,NaT,NaT,NaN,NaN,NaN,NaN,{},Deutsche Bahn,80,Germany,francfort-sur-le-main,2025-01-01 19:59:00+00:00,arrival,FR_ICE_9555_2025-01-01,False,ICE_9555_2025-01-01_8011068,False,not_flagged,intermediate,international,Deutsche Bahn,f,NaN
2245,FR,TGV INOUI,6687,2025-01-01,8772600,2025-01-01 18:44:59.511+00,TGV INOUI 6687,StopPoint:OCETGV INOUI-87726000,Saint-Étienne Châteaucreux,2025-01-01 18:48:00+00:00,2025-01-01 18:48:00+00:00,0.0,NaN,NaN,f,NaT,NaT,NaN,NaN,NaN,NaN,{},SNCF,87,France,saint-etienne-chateaucreux,2025-01-01 18:48:00+00:00,arrival,FR_TGV INOUI_6687_2025-01-01,False,TGV INOUI_6687_2025-01-01_8772600,False,not_flagged,intermediate,domestic,SNCF,f,NaN
3025,FR,TGV INOUI,8393,2025-01-01,8748500,2025-01-01 21:24:55.741+00,TGV INOUI 8393,StopPoint:OCETGV INOUI-87485003,La Rochelle,2025-01-01 21:25:00+00:00,2025-01-01 21:25:00+00:00,0.0,NaN,NaN,f,NaT,NaT,NaN,NaN,NaN,NaN,{},SNCF,87,France,la-rochelle,2025-01-01 21:25:00+00:00,arrival,FR_TGV INOUI_8393_2025-01-01,False,TGV INOUI_8393_2025-01-01_8748500,False,not_flagged,intermediate,domestic,SNCF,f,NaN


In [13]:
print(len(dep_unreliable_rows))
print(len(dep_unreliable_rows["journey_id"].unique()))

2343
2334


In [14]:
random_journey_id = dep_unreliable_rows["journey_id"].sample(1).iloc[0]
print(random_journey_id)

data_chuuchuu[data_chuuchuu["journey_id"] == random_journey_id].sort_values("sort_time")

FR_TGV INOUI_6681_2025-01-06


,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,plannedArrival,arrivalDelay,arrivalPlatform,plannedArrivalPlatform,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,operator,uicCodeStop,country,stopName_slug,sort_time,sort_time_source,journey_id,is_ambiguous_trip,journey_verificator,is_cross_agency_duplicate,cross_agency_duplicate_confidence,depart_terminus,journey_type,normalized_operator,arrivalCancelled_resolved,departureCancelled_resolved
22635,FR,TGV INOUI,6681,2025-01-06,8700012,2025-01-06 09:49:22+00,TGV INOUI 6681,StopPoint:OCETGV INOUI-87686006,Paris Gare de Lyon Hall 1 - 2,NaT,2025-01-06 06:52:00+00:00,NaN,NaN,NaN,f,2025-01-06 06:52:00+00:00,2025-01-06 06:52:00+00:00,0.0,NaN,NaN,f,"{""unplanned"": false, ""originalTripId"": ""OCESN6...",SNCF,87,France,paris-gare-de-lyon-hall-1-2,2025-01-06 06:52:00+00:00,departure,FR_TGV INOUI_6681_2025-01-06,False,TGV INOUI_6681_2025-01-06_8700012,False,not_flagged,depart,domestic,SNCF,f,f
22622,FR,TGV INOUI,6681,2025-01-06,8700167,2025-01-06 09:49:22+00,TGV INOUI 6681,StopPoint:OCETGV INOUI-87694109,Le Creusot-TGV,2025-01-06 08:12:00+00:00,2025-01-06 08:12:00+00:00,0.0,NaN,NaN,f,2025-01-06 08:15:00+00:00,2025-01-06 08:15:00+00:00,0.0,NaN,NaN,f,"{""unplanned"": false, ""originalTripId"": ""OCESN6...",SNCF,87,France,le-creusot-tgv,2025-01-06 08:12:00+00:00,arrival,FR_TGV INOUI_6681_2025-01-06,False,TGV INOUI_6681_2025-01-06_8700167,False,not_flagged,intermediate,domestic,SNCF,f,f
22633,FR,TGV INOUI,6681,2025-01-06,8700152,2025-01-06 09:49:22+00,TGV INOUI 6681,StopPoint:OCETGV INOUI-87723197,Lyon Part Dieu,2025-01-06 08:56:00+00:00,2025-01-06 08:56:00+00:00,0.0,NaN,NaN,f,2025-01-06 09:06:00+00:00,2025-01-06 09:06:00+00:00,0.0,NaN,NaN,f,"{""unplanned"": false, ""originalTripId"": ""OCESN6...",SNCF,87,France,lyon-part-dieu,2025-01-06 08:56:00+00:00,arrival,FR_TGV INOUI_6681_2025-01-06,False,TGV INOUI_6681_2025-01-06_8700152,False,not_flagged,intermediate,domestic,SNCF,f,f
22620,FR,TGV INOUI,6681,2025-01-06,8772600,2025-01-06 09:45:07.055+00,TGV INOUI 6681,StopPoint:OCETGV INOUI-87726000,Saint-Étienne Châteaucreux,2025-01-06 09:48:00+00:00,2025-01-06 09:48:00+00:00,0.0,NaN,NaN,f,NaT,NaT,NaN,NaN,NaN,NaN,{},SNCF,87,France,saint-etienne-chateaucreux,2025-01-06 09:48:00+00:00,arrival,FR_TGV INOUI_6681_2025-01-06,False,TGV INOUI_6681_2025-01-06_8772600,False,not_flagged,intermediate,domestic,SNCF,f,NaN
22634,FR,TGV INOUI,6681,2025-01-06,8700046,2025-01-06 09:49:22+00,TGV INOUI 6681,StopPoint:OCETGV INOUI-87726000,Saint-Étienne Châteaucreux,2025-01-06 09:48:00+00:00,2025-01-06 09:48:00+00:00,0.0,NaN,NaN,f,NaT,2025-01-06 09:48:00+00:00,NaN,NaN,NaN,f,"{""unplanned"": false, ""originalTripId"": ""OCESN6...",SNCF,87,France,saint-etienne-chateaucreux,2025-01-06 09:48:00+00:00,arrival,FR_TGV INOUI_6681_2025-01-06,False,TGV INOUI_6681_2025-01-06_8700046,False,not_flagged,terminus,domestic,SNCF,f,f


### Recovering more `departureCancelled` via the next stop's arrival

Some intermediate stops have a confirmed arrival (`arrivalCancelled_resolved == "f"`, i.e. the train reached the station) but a null `departureCancelled` -- so we don't know if it left. If the *next* stop on the same journey also has a confirmed arrival, the train must have departed the current stop to get there, so `departureCancelled_resolved` can be set to `"f"`.

This only targets non-terminus stops -- a terminus stop has no onward departure to resolve, and is already handled by the terminus-based inference above, so there's no overlap between the two.

In [15]:
# sort each journey's stops chronologically, then look one row ahead within the same journey
# (groupby + shift is vectorized -- no explicit loop over rows)
sorted_by_journey = data_chuuchuu.sort_values(["journey_id", "sort_time"])
next_arrival_resolved = sorted_by_journey.groupby("journey_id")["arrivalCancelled_resolved"].shift(-1)
next_stop_arrived = (next_arrival_resolved == "f").reindex(data_chuuchuu.index)

dep_inferred_from_next_stop = (
    (data_chuuchuu["depart_terminus"] != "terminus")
    & data_chuuchuu["departureCancelled"].isna()
    & (data_chuuchuu["arrivalCancelled_resolved"] == "f")
    & next_stop_arrived
)

print(f"{dep_inferred_from_next_stop.sum()} of {dep_unreliable.sum()} previously unreliable departureCancelled rows resolved via next-stop arrival")

data_chuuchuu.loc[dep_inferred_from_next_stop, "departureCancelled_resolved"] = "f"

data_chuuchuu["departureCancelled_resolved"].value_counts(dropna=False)

2317 of 2343 previously unreliable departureCancelled rows resolved via next-stop arrival


departureCancelled_resolved
f      6918127
t       117551
NaN         26
Name: count, dtype: int64

### Exporting intermediate output

This notebook resolves cancellation status at the stop level (`arrivalCancelled_resolved`, `departureCancelled_resolved`). The per-train summary -- collapsing each `journey_id` down to one row and classifying full/partial cancellation, terminus outcome and delay -- is built in nb5, which picks up this stop-level dataset.

In [16]:
export_data_chuuchuu = input("Export cancellation data to parquet? (y/n): ")

if export_data_chuuchuu.lower() == "y":
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    cancellations_export_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_cancellations.parquet"
    data_chuuchuu.to_parquet(cancellations_export_path)
    print(f"Saved to {cancellations_export_path}")

Saved to intermediate_outputs/data_chuuchuu_french_cancellations.parquet


In [17]:
data_chuuchuu["normalized_operator"].value_counts(dropna=False)

normalized_operator
SNCF                                       6420840
SNCF VOYAGEURS                              337966
Eurostar                                    153926
Deutsche Bahn                                62093
Trenitalia                                   24845
SNCF Voyageurs LO                            21041
OCEdefault                                   12469
Conseil Régional Auvergne - Rhône-Alpes       1727
SNCF Voyageurs EA                              718
SNCF Voyageurs SA                               79
Name: count, dtype: int64

In [18]:
data_chuuchuu[data_chuuchuu["arrivalCancelled_resolved"]=="t"]["operator"].unique()

<ArrowStringArray>
[                                   'SNCF',
                                'Eurostar',
                           'Deutsche Bahn',
                          'SNCF VOYAGEURS',
 'Conseil Régional Auvergne - Rhône-Alpes',
                       'SNCF Voyageurs LO',
                       'SNCF Voyageurs SA']
Length: 7, dtype: str

In [19]:
data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="t"]["operator"].unique()

<ArrowStringArray>
[                                   'SNCF',
                                'Eurostar',
                           'Deutsche Bahn',
                          'SNCF VOYAGEURS',
 'Conseil Régional Auvergne - Rhône-Alpes',
                       'SNCF Voyageurs LO',
                       'SNCF Voyageurs SA']
Length: 7, dtype: str